In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sb

In [ ]:
total_rainfall = pd.read_csv("rainfall-monthly-total.csv")
no_of_rainy_days = pd.read_csv("rainfall-monthly-number-of-rain-days.csv")
maximum_rainfall_in_day = pd.read_csv("rainfall-monthly-highest-daily-total.csv")

### Observation
- We need data handling (pandas, numpy) and plotting (matplotlib, seaborn).

### Decision
- Import the basic libraries first. The modelling libraries are imported later, just before they are needed.

### Result
- Libraries loaded successfully (no output expected).



In [ ]:
total_rainfall.head()

In [ ]:
no_of_rainy_days.head()

In [ ]:
maximum_rainfall_in_day.head()

### Observation
- The information is split across three separate CSV files.
- Each file is monthly data with a month column (format YYYY-MM) and one value column.

### Decision
- Use month as the common key to combine them into a single table.

### Result
- Three DataFrames loaded. Each shows month plus one rainfall measure (total_rainfall, no_of_rainy_days, maximum_rainfall_in_a_day).


In [ ]:
total_rainfall.info()

In [ ]:
no_of_rainy_days.info()

In [ ]:
maximum_rainfall_in_day.info()

In [ ]:
data = total_rainfall.merge(maximum_rainfall_in_day, on="month").merge(no_of_rainy_days, on= "month")

### Observation
- We checked missing values, duplicate rows and the shape (rows x columns).
- Time-series models (SARIMAX, LSTM) give wrong results or fail if values are missing.

### Decision
- If null counts are all 0 and duplicated prints False, no cleaning is needed.
- Otherwise, fill missing months and drop duplicates before continuing.

### Result
- Null counts per column, True/False for duplicates, and the (rows, 4) shape are shown above.
- The code continues without a cleaning step, so it expects a clean dataset.

In [ ]:
data.head()

In [ ]:
data.isnull().sum()

In [ ]:
print(data.duplicated().any())

In [ ]:
data.shape

### Observation
- We checked missing values, duplicate rows and the shape (rows x columns).
- Time-series models (SARIMAX, LSTM) give wrong results or fail if values are missing.

### Decision
- If null counts are all 0 and duplicated prints False, no cleaning is needed.
- Otherwise, fill missing months and drop duplicates before continuing.

### Result
- Null counts per column, True/False for duplicates, and the (rows, 4) shape are shown above.
- The code continues without a cleaning step, so it expects a clean dataset.


In [ ]:
data["month"] = pd.to_datetime(data["month"], format= "%Y-%m")

data = data.set_index("month")
data = data.sort_index()

### Observation
- Forecasting needs data in chronological order with a date index.
- The raw month column was text. The value columns should be numeric (float or int).

### Decision
- Convert month to datetime, set it as the index, and sort ascending.
- Confirm numeric dtypes, because an object column would crash the models.

### Result
- The index is now a DatetimeIndex sorted oldest to newest.
- No nulls were introduced by the conversion, and the dtype check confirms the value columns are numeric.

In [ ]:
data.head()

In [ ]:
data.isnull().sum()

In [ ]:
data.info()

In [ ]:
data["maximum_rainfall_in_a_day"].dtype

In [ ]:
# EDA

In [ ]:
cols = ["total_rainfall", "maximum_rainfall_in_a_day", "no_of_rainy_days"]

plt.figure(figsize = (10,3))

for i,  col in enumerate(cols, start = 1):
  plt.subplot(1,3,i)
  sb.histplot(data  = data[col], kde = True)
  plt.title(col)
plt.suptitle("HISTPLOT" ,fontsize = 15,fontweight ="bold")
plt.tight_layout()
plt.show()

### Observation
- Histograms with a KDE curve show how each variable is spread.
- Rainfall variables are normally right-skewed: many ordinary months and a few very wet months.
- no_of_rainy_days is a count limited to 31, so it looks more symmetric.

### Decision
- Scale the data with MinMaxScaler for the neural networks.
- Clip final forecasts at 0, because rainfall cannot be negative.

### Result
- Three distribution plots produced. Check the long right tail on the rainfall columns.

In [ ]:
plt.figure(figsize = (15, 5))

sb.pairplot(data)
plt.suptitle("PAIRPLOT" ,fontsize = 15,fontweight ="bold")
plt.tight_layout()
plt.show()


### Observation
- The pairplot shows every variable against every other one.
- Total rainfall and number of rainy days are expected to rise together.
- Heavy single-day rain is expected to push the monthly total up.

### Decision
- Use it only to understand the data.
- The final models are univariate: each target is predicted from its own past values.

### Result
- A 3x3 grid of scatter plots and distributions. Upward trends confirm positive relationships.

In [ ]:
# BOXPLOT


plt.figure(figsize=(15,5))
plt.suptitle("BOXPLOT" ,fontsize = 15,fontweight ="bold")
for i,col in enumerate(cols, start = 1):
  plt.subplot(1,3,i)
  sb.boxplot(data = data[col])
  plt.title(col)
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()


### Observation
- Points beyond the whiskers are outliers.
- For rainfall these are usually real extreme-weather months, not errors.

### Decision
- Do not remove the outliers. They are genuine heavy-rain events, and removing them would make the forecast under-predict extreme rain.

### Result
- Boxplots drawn for all three variables. Outliers are mainly at the top (very wet months).

In [ ]:
plt.title("HEATMAP")
sb.heatmap(data.corr(), annot = True)
plt.show()


### Observation
- The heatmap gives the correlation (-1 to +1) between each pair of variables.
- Rainy days and total rainfall are expected to be positively correlated.

### Decision
- Correlations are informational only.
- Keep the univariate time-series approach and focus on the two targets: total_rainfall and maximum_rainfall_in_a_day.

### Result
- A 3x3 annotated correlation matrix. Values close to +1 indicate a strong positive relationship.

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense, Input, LSTM

from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import MinMaxScaler

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

### Observation
- Three model families are compared: a statistical model (SARIMAX) and two deep-learning models (SimpleRNN, LSTM).

### Decision
- Use RMSE as the metric. It is in the same unit as rainfall (mm), so it is easy to interpret.

### Result
- Libraries imported. TensorFlow may print info or warning messages, which are harmless.

In [ ]:
WINDOW_SIZE = 12
def arima_forecast(train_data, months_to_predict):

    model = SARIMAX(
        train_data,
        order=(1,0,1),
        seasonal_order=(1,1,1,12)
    )

    model = model.fit(disp=False)
    prediction = model.forecast(months_to_predict)

    return np.array(prediction)

### Observation
- Rainfall has a strong yearly (12-month) seasonality, so a seasonal model is a natural fit.

### Decision
- order=(1,0,1) for the short-term pattern and seasonal_order=(1,1,1,12) for the yearly pattern.
- WINDOW_SIZE = 12 is reused for the neural networks: look back one full year.

### Result
- Function arima_forecast defined. It returns a NumPy array of the next N months.

In [ ]:
def rnn_forecast(train_data, months_to_predict):
  scaler = MinMaxScaler()
  scaled_data = scaler.fit_transform(train_data.values.reshape(-1,1))
  x = []
  y = []


  for i in range(len(scaled_data) - WINDOW_SIZE):
    x.append(scaled_data[i:i+WINDOW_SIZE])
    y.append(scaled_data[i+WINDOW_SIZE])

  x = np.array(x)
  y = np.array(y)

  x = x.reshape(x.shape[0], WINDOW_SIZE, 1)

  model = Sequential()
  model.add(Input(shape=(WINDOW_SIZE,1)))
  model.add(SimpleRNN(16))
  model.add(Dense(1))

  model.compile(
      optimizer = "adam",
      loss = "mse"
  )

  model.fit(x,y, epochs = 50, verbose = 0, batch_size = 32)

  previous_values = list(scaled_data[-WINDOW_SIZE:].flatten())
  prediction = []

  for i in range(months_to_predict):
    x_input = np.array(previous_values[-WINDOW_SIZE:]).reshape(1, WINDOW_SIZE, 1)
    val = model.predict(x_input, verbose=0)
    prediction.append(val[0][0])
    previous_values.append(val[0][0])

  prediction = scaler.inverse_transform(np.array(prediction).reshape(-1,1))

  return prediction

### Observation
- Neural networks need scaled data and input windows: 12 past months to predict the next month.
- Forecasting 12 months ahead is done recursively: each prediction is fed back as input for the next step.

### Decision
- Scale to 0-1 with MinMaxScaler, build sliding windows, train a small SimpleRNN(16) for 50 epochs, then inverse-transform the output back to mm.

### Result
- Function rnn_forecast defined. It returns a (months, 1) array in original rainfall units.
- Recursive forecasting can accumulate error, so later months are less reliable.


In [ ]:
def lstm_forecast(train_data, months_to_predict):
  x = []
  y = []

  scaler = MinMaxScaler()
  scaled_data = scaler.fit_transform(train_data.values.reshape(-1,1))


  for i in range(len(scaled_data) - WINDOW_SIZE):
    x.append(scaled_data[i:i+WINDOW_SIZE])
    y.append(scaled_data[i +WINDOW_SIZE])
  x = np.array(x)
  y = np.array(y)

  x = x.reshape(x.shape[0], WINDOW_SIZE, 1)

  model = Sequential()
  model.add(Input(shape=(WINDOW_SIZE,1)))
  model.add(LSTM(16))
  model.add(Dense(1))

  model.compile(
      optimizer = "adam",
      loss = "mse"
  )

  model.fit(x,y, verbose = 0, epochs = 50, batch_size = 32)

  previous_values = list(scaled_data[-WINDOW_SIZE:].flatten())
  prediction = []

  for i in range(months_to_predict):
    x_input = np.array(previous_values[-WINDOW_SIZE:]).reshape(1, WINDOW_SIZE, 1)

    val = model.predict(x_input, verbose=0)
    prediction.append(val[0][0])
    previous_values.append(val[0][0])

  prediction = scaler.inverse_transform(np.array(prediction).reshape(-1,1))
  return prediction

In [ ]:

def run_all_models(train_data, months_to_predict):

    results = {}

    results["ARIMA"] = arima_forecast(train_data, months_to_predict)

    results["RNN"] = rnn_forecast(train_data, months_to_predict)

    results["LSTM"] = lstm_forecast(train_data, months_to_predict)

    return results

future_months = pd.date_range(
    data.index[-1] + pd.offsets.MonthBegin(1),
    periods=12,
    freq="MS"
)

final_forecast = {}

targets = [
    "total_rainfall",
    "maximum_rainfall_in_a_day"
]

for target in targets:

    print("\n" + "=" * 50)
    print("TARGET :", target)
    print("=" * 50)

    series = data[target]

    train_data = series[:-12]

    test_data = series[-12:]

    predictions = run_all_models(train_data, 12)

    errors = {}

    for model_name in predictions:

        rmse = np.sqrt(
            mean_squared_error(
                test_data,
                predictions[model_name]
            )
        )

        errors[model_name] = rmse

        print(model_name, "RMSE :", round(rmse,2))

    best_model = min(errors, key=errors.get)

    print("Best Model :", best_model)

    future_prediction = run_all_models(series,12)

    final_forecast[target] = np.clip(
        future_prediction[best_model],
        0,
        None
    )

    plt.figure(figsize=(10,4))

    plt.plot(
        series[-48:],
        label="Actual Data"
    )

    plt.plot(
        future_months,
        final_forecast[target],
        color="red",
        label="Forecast"
    )

    plt.title(target)

    plt.xlabel("Month")

    plt.ylabel("Rainfall")

    plt.legend()

    plt.show()


result = pd.DataFrame({

    "Month": future_months.strftime("%b %Y"),

    "Total Rainfall (mm)": final_forecast["total_rainfall"].round(1),

    "Highest Rainfall In A Day (mm)": final_forecast["maximum_rainfall_in_a_day"].round(1)

})

print("\nNEXT YEAR RAINFALL FORECAST\n")

print(result.to_string(index=False))

wettest_month = result.loc[
    result["Total Rainfall (mm)"].idxmax(),
    "Month"
]

highest_day = result.loc[
    result["Highest Rainfall In A Day (mm)"].idxmax(),
    "Month"
]

print("\nWettest Month :", wettest_month)

print("Highest One-Day Rainfall Month :", highest_day)

result.to_csv(
    "next_year_forecast.csv",
    index=False
)

### Observation
- Useful questions to answer: which month will be the wettest, and which month will have the heaviest single-day rain.

### Decision
- Take the row with the maximum value in each column (idxmax) and save the whole table to CSV for sharing.

### Result
- The wettest month and the highest one-day rainfall month are printed.
- next_year_forecast.csv is saved in the working folder.